In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from scipy import stats


In [ ]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()

In [ ]:


X = housing.drop("median_house_value", axis=1)
y = housing["median_house_value"]


print(X.head())
print(y.head())
print(X.shape)
print(X.dtypes)


In [ ]:
X.fillna(0)

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

X = pd.get_dummies(X.fillna(0))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

initial_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Initial RMSE: {initial_rmse}")

# Cross-validation
scores = cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=5)

# Calculate the root mean squared error (RMSE) for each fold
rmse_scores = np.sqrt(-scores)

# Output the results
print("Root Mean Squared Error (RMSE) scores for each fold:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Standard Deviation of RMSE:", rmse_scores.std())

In [ ]:
housing = load_housing_data()

In [ ]:
X = housing.drop("median_house_value", axis=1)
y = housing["median_house_value"]

In [ ]:
# Get numeric outliers
X_numeric = X.select_dtypes(include="number")

# Remove Outliers
Q1 = X_numeric.quantile(0.25)
Q3 = X_numeric.quantile(0.75)
IQR = Q3 - Q1

# Determine a mask for rows without outliers
mask = ~((X_numeric < (Q1 - 1.5 * IQR)) | (X_numeric > (Q3 + 1.5 * IQR))).any(axis=1)
X_clean = X[mask]
y_clean = y[mask]

print(f"Shape before cleaning: {X.shape}")
print(f"Shape after cleaning: {X_clean.shape}")


In [ ]:
# Numeric attributes pipeline
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('scaler', StandardScaler()),
])

# Categorical attributes pipeline
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

# Combine numerical and categorical pipelines
preprocessor = ColumnTransformer([
    ("num", num_pipeline, ["longitude", "latitude", "housing_median_age", "total_rooms",
                           "total_bedrooms", "population", "households", "median_income"]),
    ("cat", cat_pipeline, ["ocean_proximity"]),
])

X_clean = preprocessor.fit_transform(X_clean)


In [ ]:
# Train on cleaned data
X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train_clean, y_train_clean)
y_pred = model.predict(X_test_clean)

cleaned_rmse = np.sqrt(mean_squared_error(y_test_clean, y_pred))
print(f"Cleaned Data RMSE: {cleaned_rmse}")

# Cross-validation
scores = cross_val_score(model, X_clean, y_clean, scoring='neg_mean_squared_error', cv=5)

# Calculate the root mean squared error (RMSE) for each fold
rmse_scores = np.sqrt(-scores)

# Output the results
print("Root Mean Squared Error (RMSE) scores for each fold:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Standard Deviation of RMSE:", rmse_scores.std())